Para a minha implementação, tive a ideia de validar o cadastro básico de um produto de uma lojinha. O objetivo é mockar os dados de forma que eu apenas precise validar usando Pydantic. Não me aprofundarei em FastAPI, apenas na validação de dados.

In [13]:
import enum
import re
from typing import Any

from pydantic import (
    BaseModel, Field, field_validator, model_validator, ValidationError,
)

VALID_SKU_REGEX = re.compile(r"^[A-Z]{3}-\d{4}$")

Aqui, importei enum para enumerar os itens do estoque, Any para receber o JSON e as funções que serão utilizadas do Pydantic.
Seguindo o mesmo padrão dos exemplos, usarei Regex para validar os nomes.

In [14]:
class Category(enum.IntFlag):
    Eletronicos = 1
    Vestuarios = 2
    Alimento = 4

Nessa classe eu defino as Roles de cada item que pode entrar no estoque. O uso do `IntFlag` é justificado dado que um valor sempre vai ser interpretado como a role que foi atribuída a esse valor, garantindo previsibilidade.

In [15]:
class Product(BaseModel):
    name: str = Field(examples=["Notebook Dell"])
    sku: str = Field(
        examples=["DEL-9080"],
        description = "Código único do produto",
        frozen=True,
    )
    cost_price: float = Field(
        description="Preço pago pelo fornecedor", exclude=True
    )
    sale_price: float = Field(description="Preço ofertado para o consumidor")
    category: Category=Field(
        default = Category.Eletronicos, description="Categoria do produto"
    )

Aqui, estou implementando a classe Produto, cujo é o um produto mockado que passará pela validação posteriormente. O produto possui SKU, que é um código único e imutável do produto, cost_price, que é o preço de compra, sale_price, preço de venda, e a sua categoria que varia de acordo com a role especificada.

Nota-se o `exclude=True` no cost_price. Isso ocorre dado que é interessante para uma loja esconder o valor do seu preço de compra e assim esconder demais informações que poderiam ser relevantes para concorrentes.

In [16]:
@field_validator("sku")
@classmethod
def validate_sku(cls, v: str) -> str:
    if not VALID_SKU_REGEX.match(v):
        raise ValueError(
            "Sku inválido, deve conter 3 letras maiúsculas, um travessão e 4 algarismos"
        )
    return v

@model_validator(mode="after")
def validate_profit(self):
    if self.sale_price <= self.cost_price:
        raise ValueError("O preço de venda deve ser estritamente maior que o preço de compra")
    return self


Basicamente, aqui usamos das expressões regulares para valiar o SKU e avalio se o preço de venda foi menor que o preço de compra (coisa que nã0 pode acontecer).

In [18]:
def validate(data: dict[str, Any]) -> None:
    try:
        product = Product.model_validate(data)
        print(product)
    except ValidationError as e:
        print("Produto inválido")
        print(e)

def main() -> None:
    test_data = dict(
        good_product={...},
        bad_sku_formula={...},
        bad_profit_logic={...},
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)
        print()


if __name__ == "__main__":
    main()

good_product
Produto inválido
1 validation error for Product
  Input should be a valid dictionary or instance of Product [type=model_type, input_value={Ellipsis}, input_type=set]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

bad_sku_formula
Produto inválido
1 validation error for Product
  Input should be a valid dictionary or instance of Product [type=model_type, input_value={Ellipsis}, input_type=set]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

bad_profit_logic
Produto inválido
1 validation error for Product
  Input should be a valid dictionary or instance of Product [type=model_type, input_value={Ellipsis}, input_type=set]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type



failed to send, dropping 1 traces to intake at http://localhost:8126/v0.5/traces: client error (Connect) [1 skipped]


Em resumo, aqui eu valido o produto criado e faço testes personalizados na main. No geral, a lógica do código se assemelha à lógica dos exemplos apresentados. Foi uma implementação realmente simples, mas reveladora quanto à usabilidade do Pydantic e de como funcionam as validações, serialização e desserialização em Python.